# This notebook does four things:
* necessary imports
* loading input data
* calling training function
* saving metrics

## Imports and parameters

In [ ]:
import pandas as pd
from pathlib import Path
import pickle
import glob
import os
import matplotlib.pyplot as plt

from utils.model_functions import model_training


In [ ]:
target_column = "kWh_received_Total"
time_column = 'Timestamp'

horizon = 24 * 4  # one day in 15 min time resolution

n_outer_splits = 5
n_inner_splits = 2
random_state = 42

## Input data

### Loading

In [ ]:
file_path = Path("data/interim/household_weather.parquet")
# inputs_df = pd.read_csv(file_path)
inputs_df = pd.read_parquet(file_path)
inputs_df.drop(columns=['Household_ID'], inplace=True)

# print(inputs_df.describe())

## Model training

In [ ]:
for model_type in ['xgboost']: #'linearregression', 'ridge', 'elasticnet', 
    print(f"\nTraining model: {model_type}\n")
    model_training(inputs_df,
                   time_column=time_column,
                   target_column=target_column,
                   model_type=model_type,
                   n_outer_splits=n_outer_splits,
                   n_inner_splits=n_inner_splits,
                   random_state=random_state)

In [ ]:
results = {}
model_type = 'xgboost'
results[model_type] = {}
results_path = Path(f"results/estimated_production_metrics_{model_type}.pkl")
results_df = pd.read_pickle(results_path)
results[model_type]['mean_rmse'] = results_df['rmse'].mean()
results[model_type]['std_rmse'] = results_df['rmse'].std()

In [ ]:
results = {}

# Find all saved pickle files in the results directory
file_paths = glob.glob("results/estimated_production_metrics_*.pkl")

for file_path in file_paths:
    # Extract the model_type from the filename (e.g., "xgboost")
    filename = os.path.basename(file_path)
    model_type = filename.replace("estimated_production_metrics_", "").replace(".pkl", "")
    
    # Open and load the dictionary from the pickle file
    with open(file_path, "rb") as f:
        results[model_type] = pickle.load(f)

# --- Your existing plotting code below ---

models = list(results.keys())
mean_rmses = [results[model]['mean_rmse'] for model in models]
std_rmses = [results[model]['std_rmse'] for model in models]

plt.figure(figsize=(10, 6))
plt.bar(models, mean_rmses, yerr=std_rmses, capsize=5)
plt.xlabel("Model type")
plt.ylabel("RMSE")
plt.title("Model Performance")
fig = plt.gcf()
fig.savefig("results/model_performance_RMSE.png", dpi=300)
plt.show()

In [ ]:
model_type = 'xgboost'
model_file = Path(f"results/estimated_production_predictions_{model_type}.pkl")
oof_predictions_df = pd.read_pickle(model_file).dropna()

plt.figure(figsize=(15, 6))
plt.plot(oof_predictions_df['time'], oof_predictions_df['y_true'], label='Observation', color='blue')
plt.plot(oof_predictions_df['time'], oof_predictions_df['y_pred'], label='Prediction', color='orange')
plt.xlabel('Time')
plt.ylabel('kWh')
plt.title('Observed vs. Forecasted Values')
plt.legend()
fig = plt.gcf()
fig.savefig("results/observed_vs_forecasted.png", dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_true, y_pred, alpha=0.3, color='blue', label='Predictions')

# Add a 1:1 reference line (perfect prediction)
max_val = max(y_true.max(), y_pred.max())
min_val = min(y_true.min(), y_pred.min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', label='Perfect Fit')

plt.xlabel('Observed values (kWh)')
plt.ylabel('Predicted values (kWh)')
plt.title('Observed vs. predicted values')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

fig = plt.gcf()
fig.savefig("results/scatter_predicted_vs_observed.png", dpi=300)
plt.show()